## Step by Step RAG

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

llm_model = "gpt-4.1-nano"

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

/Volumes/data/wk/sudhir4ev/dlai/dlai/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [5]:
loader = CSVLoader(file_path="assets//OutdoorClothingCatalog_1000.csv")
docs = loader.load()

print(docs[0])

page_content=': 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.' metadata={'source': 'assets//OutdoorClothingCatalog_1000.csv', 'row': 0}


### Create Embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
embed = embeddings.embed_query("Hi my name is Harrison")
print(len(embed))
print(embed[:5])

1536
[0.005666269920766354, -0.005918576847761869, -0.06274238973855972, 0.022665079683065414, -0.05136121064424515]


#### Create Vector Store

In [9]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_documents(
    docs,
    embedding=embeddings
)

#### Similarity search

In [11]:
query = "Please suggest a shirt with sunblocking"

results = vectorstore.similarity_search(query)
len(results)
print(results[0].page_content)

: 255
name: Sun Shield Shirt by
description: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. 

Size & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.

Fabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the highest rated sun protection possible. Handwash, line dry.

Additional Features: Wicks moisture for quick-drying comfort. Fits comfortably over your favorite swimsuit. Abrasion resistant for season after season of wear. Imported.

Sun Protection That Won't Wear Off
Our high-performance fabric provides SPF 50+ sun protection, blocking 98% of the sun's harmful rays. This fabric is recommended by The Skin Cancer Foundation as an effective UV protectant.


#### Create Retriever

In [12]:
retriever = vectorstore.as_retriever()

#### Mordern LLM Setup

In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=llm_model,
    temperature=0.0
)

Create prompt

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Return the results in a markdown table and summarize each shirt.
""")

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

Build Retrieval chain

In [17]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

Execute query

In [19]:
from IPython.display import Markdown, display

query = "Please list all your shirts with sun protection in a table in markdown and summarize each one."

response = chain.invoke(query)

display(Markdown(response.content))

| Shirt Name                                | Summary                                                                                                              |
|--------------------------------------------|----------------------------------------------------------------------------------------------------------------------|
| Sun Shield Shirt by                        | High-performance sun shirt with UPF 50+ protection, blocking 98% of UV rays. Softly fitted, falls at hip, moisture-wicking, abrasion-resistant, imported. Designed to protect from sun without sacrificing fun. |
| Men's Tropical Plaid Short-Sleeve Shirt   | Light, relaxed fit shirt rated UPF 50+ with 98% UV protection. Made of 100% polyester, wrinkle-resistant, with venting and pockets, ideal for hot weather and sun protection. |
| Men's Plaid Tropic Shirt, Short-Sleeve    | Ultracomfortable, UPF 50+ rated shirt blocking 98% UV rays. Made of 52% polyester and 48% nylon, quick-drying, wrinkle-free, with venting and pockets, suitable for extended travel. |
| Tropical Breeze Shirt                      | Lightweight, breathable long-sleeve shirt with UPF 50+ protection, moisture-wicking, wrinkle-resistant, designed for outdoor activities like fishing and travel. |